In [305]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [306]:
import os
import json
import numpy as np
import pandas as pd
from typing import Dict, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

In [307]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")

# ID mappings (from 02_id_encoder.ipynb)
id_mappings_dir = os.path.join(preprocessed_dir, "id_mappings")
user_id_2idx_path = os.path.join(id_mappings_dir, "user_id_2idx.json")
item_id_2idx_path = os.path.join(id_mappings_dir, "item_id_2idx.json")
scene_2idx_path = os.path.join(id_mappings_dir, "scene_2idx.json")
id_vocab_config_path = os.path.join(id_mappings_dir, "id_vocab_config.json")

# Precomputed embeddings
text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")               # (N_items, 768)
image_embeddings_aligned_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")              # (N_interactions, D_time)
time_feature_cols_path = os.path.join(preprocessed_dir, "time_features_columns.json")      # list of column names

# fusion artifacts from this notebook will be saved here
fusion_artifacts_dir = os.path.join(base_dir, "fusion_artifacts")
os.makedirs(fusion_artifacts_dir, exist_ok=True)

In [308]:
# sanity check
print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("id_mappings_dir:", id_mappings_dir)
print("text_embeddings_path:", text_embeddings_path)
print("image_embeddings_path:", image_embeddings_aligned_path)
print("time_features_path:", time_features_path)
print("fusion_artifacts_dir:", fusion_artifacts_dir)

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
id_mappings_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/id_mappings
text_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/text_embeddings.npy
image_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_embeddings_vit_base_aligned.pt
time_features_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/time_features_final.pt
fusion_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts


In [309]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [310]:
# Next, we will load Embedding Sizes (Dynamic Sizing)
# Load shapes without pulling everything into GPU memory
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
text_dim = text_embeddings.shape[1]

image_embeddings = torch.load(image_embeddings_aligned_path, map_location="cpu")
image_dim = image_embeddings.shape[1]

time_features = torch.load(time_features_path, map_location="cpu")
time_raw_dim = time_features.shape[1]

In [311]:
# Load time feature names
with open(time_feature_cols_path, "r") as f:
    time_feature_cols = json.load(f)

print("--- Embedding/Feature Dimensions ---")
print("Text embedding dim:", text_dim)
print("Image embedding dim:", image_dim)
print("Time raw feature dim:", time_raw_dim)
print("Time feature columns:", time_feature_cols)

--- Embedding/Feature Dimensions ---
Text embedding dim: 1536
Image embedding dim: 768
Time raw feature dim: 5
Time feature columns: ['unix_time_norm', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend']


In [312]:
# Load vocab sizes (for ID embeddings)
with open(id_vocab_config_path, "r") as f:
    id_vocab_config = json.load(f)

user_vocab_size = int(id_vocab_config["num_users"])
item_vocab_size = int(id_vocab_config["num_items"])
scene_vocab_size = int(id_vocab_config["num_scenes"])

print("\n--- ID Vocabulary Sizes ---")
print("user_vocab_size:", user_vocab_size)
print("item_vocab_size:", item_vocab_size)
print("scene_vocab_size:", scene_vocab_size)


--- ID Vocabulary Sizes ---
user_vocab_size: 67615
item_vocab_size: 59142
scene_vocab_size: 5


## Load All Precomputed Embeddings / Feature Shapes

In this step, we validate and summarize the shapes of all modality inputs that will enter the ColdFusionX fusion module.
These inputs reference back to earlier notebooks:
*   Text embeddings from 03_text_encoder_BERT.ipynb
*   Image embeddings from 04_image_encoder.ipynb (ViT-base CLS, 768-d)
*   Time features from 05_time_encoder.ipynb (raw time vector, 5-d)
*   ID vocab sizes from 02_id_encoder.ipynb (user/item/scene mappings)

To keep memory usage low, we first load arrays/tensors in a lightweight way (e.g., NumPy memmap, CPU tensors).
We then run sanity checks to ensure dimensions match expected vocab sizes and that indexing will remain consistent during model training.

In [313]:
# Lightweight load
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")  # interaction-level
image_embeddings = torch.load(image_embeddings_aligned_path, map_location="cpu")  # item-level aligned
time_features = torch.load(time_features_path, map_location="cpu")  # interaction-level

num_text_rows, text_dim = text_embeddings.shape
num_image_rows, image_dim = image_embeddings.shape
num_time_rows, time_raw_dim = time_features.shape

In [314]:
print("- Precomputed Embeddings / Feature Shapes ")
print(f"Text embeddings:  shape = ({num_text_rows}, {text_dim})  [interaction-level]")
print(f"Image embeddings: shape = ({num_image_rows}, {image_dim}) [item-level aligned]")
print(f"Time features:    shape = ({num_time_rows}, {time_raw_dim}) [interaction-level]")

- Precomputed Embeddings / Feature Shapes 
Text embeddings:  shape = (3730329, 1536)  [interaction-level]
Image embeddings: shape = (59142, 768) [item-level aligned]
Time features:    shape = (3730329, 5) [interaction-level]


In [315]:
# Load time feature names
with open(time_feature_cols_path, "r") as f:
    time_feature_cols = json.load(f)

print("\nTime feature columns:", time_feature_cols)


Time feature columns: ['unix_time_norm', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend']


In [316]:
# Load missing mask
image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")
print("Missing image embeddings (count):", image_missing_mask.sum().item())

Missing image embeddings (count): 2


In [317]:
# item-level alignment (image embeddings)
assert num_image_rows == item_vocab_size, (
    f"Mismatch: image_embeddings rows ({num_image_rows}) != item_vocab_size ({item_vocab_size})"
)

# interaction-level alignment (text embeddings + time features)
assert num_text_rows == num_time_rows, (
    f"Mismatch: text rows ({num_text_rows}) != time rows ({num_time_rows})"
)

# time feature column alignment
assert time_raw_dim == len(time_feature_cols), (
    f"Mismatch: time_raw_dim ({time_raw_dim}) != number of time feature columns ({len(time_feature_cols)})"
)

In [318]:
# Basic numerical checks
# Sample small slices to avoid scanning all 3.7M rows
sample_idx = np.random.randint(0, num_time_rows, size=2048)

text_sample = np.array(text_embeddings[sample_idx])
time_sample = time_features[sample_idx].numpy()
image_sample = image_embeddings[np.random.randint(0, num_image_rows, size=256)].numpy()

assert not np.isnan(text_sample).any(), "NaNs found in text embeddings sample."
assert not np.isnan(time_sample).any(), "NaNs found in time features sample."
assert not np.isnan(image_sample).any(), "NaNs found in image embeddings sample."

print("Sanity checks passed (item-level + interaction-level alignment confirmed).")

Sanity checks passed (item-level + interaction-level alignment confirmed).


### FeatureStore Implementation

This step implements a lightweight FeatureStore that centralizes access to all precomputed modality features:

*   Text embeddings are loaded via NumPy memmap to avoid RAM blow-up.
*   Image embeddings are loaded as a CPU tensor (aligned to full item vocabulary).
*   Time features are loaded as a CPU tensor (moved to GPU per batch during training).
*   ID triplets (user/item/scene indices) are loaded from the preprocessed df_id_encoded file.

The FeatureStore provides a consistent retrieval API used by CTRDataset, fusion modules, training, evaluation, and explainability.

In [358]:
# First we will Load df_id_encoded + validate required columns
# Prefer pickle (faster, preserves types), fallback to CSV
df_id_encoded_pkl = os.path.join(preprocessed_dir, "split_features", "df_id_encoded.pkl")
df_id_encoded_csv = os.path.join(preprocessed_dir, "split_features", "df_id_encoded.csv")

if os.path.exists(df_id_encoded_pkl):
    df_id = pd.read_pickle(df_id_encoded_pkl)
    print("Loaded df_id_encoded.pkl:", df_id.shape)
else:
    df_id = pd.read_csv(df_id_encoded_csv)
    print("Loaded df_id_encoded.csv:", df_id.shape)

# --- REQUIRED columns (must exist) ---
required_cols = ["user_id_idx", "item_id_idx", "scene_idx"]
missing_cols = [c for c in required_cols if c not in df_id.columns]

print("df_id columns:", list(df_id.columns))
assert len(missing_cols) == 0, f"Missing required columns in df_id_encoded: {missing_cols}"

# label column (different pipelines may name it differently)
label_col_candidates = ["label", "is_click", "click", "y"]
label_col = next((c for c in label_col_candidates if c in df_id.columns), None)

print("Detected label column:", label_col)
df_id.head()

Loaded df_id_encoded.pkl: (3730329, 3)
df_id columns: ['user_id_idx', 'item_id_idx', 'scene_idx']
Detected label column: None


,user_id_idx,item_id_idx,scene_idx
0,41688,13590,4
1,39834,21351,2
2,22354,33398,2
3,17547,11190,1
4,55873,33398,2


In [320]:
class FeatureStore:
    """
    Centralized feature access layer for ColdFusionX.

    - text_embeddings: interaction-level (row_idx aligned)
    - time_features: interaction-level (row_idx aligned)
    - image_embeddings: item-level (item_idx aligned)
    - id mappings: optional (debug/traceability)
    """
    def __init__(
        self,
        text_embeddings_path: str,
        image_embeddings_path: str,
        time_features_path: str,
        df_id: pd.DataFrame,
        image_missing_mask_path: Optional[str] = None,
        id_maps: Optional[Dict[str, Dict]] = None
    ):
        # Interaction-level embeddings (memmap)
        self.text_embeddings = np.load(text_embeddings_path, mmap_mode="r")  # (N_rows, text_dim)

        # Item-level embeddings (aligned)
        self.image_embeddings = torch.load(image_embeddings_path, map_location="cpu")  # (num_items, 768)

        # Interaction-level time features
        self.time_features = torch.load(time_features_path, map_location="cpu")  # (N_rows, time_dim)

        # ID index dataframe (interaction-level)
        self.df_id = df_id.reset_index(drop=True)

        # Optional mask: True indicates missing image embedding
        self.image_missing_mask = None
        if image_missing_mask_path and os.path.exists(image_missing_mask_path):
            self.image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")

        # ID mapping dicts (debugging)
        self.id_maps = id_maps or {}

        # Cache shapes for quick checks
        self.num_rows = len(self.df_id)
        self.text_dim = self.text_embeddings.shape[1]
        self.image_dim = self.image_embeddings.shape[1]
        self.time_dim = self.time_features.shape[1]

        # Strong alignment checks (interaction-level)
        assert self.text_embeddings.shape[0] == self.num_rows, (
            f"text_embeddings rows ({self.text_embeddings.shape[0]}) != df_id rows ({self.num_rows})"
        )
        assert self.time_features.shape[0] == self.num_rows, (
            f"time_features rows ({self.time_features.shape[0]}) != df_id rows ({self.num_rows})"
        )

    # --------------
    # API Methods
    # --------------
    def get_text(self, row_idx: int) -> np.ndarray:
        """Return interaction-level text vector for a given row index."""
        return np.array(self.text_embeddings[row_idx], dtype=np.float32)

    def get_time(self, row_idx: int) -> torch.Tensor:
        """Return interaction-level time features for a given row index."""
        return self.time_features[row_idx].float()

    def get_id_triplet(self, row_idx: int) -> Tuple[int, int, int]:
        """Return (user_id_idx, item_id_idx, scene_idx) for a given row index."""
        row = self.df_id.iloc[row_idx]
        return int(row["user_id_idx"]), int(row["item_id_idx"]), int(row["scene_idx"])

    def get_image(self, item_id_idx: int) -> torch.Tensor:
        """Return item-level image embedding for a given item index (aligned)."""
        return self.image_embeddings[item_id_idx].float()

    # -----------------------
    # Additional helpers
    # -----------------------
    def has_image(self, item_id_idx: int) -> bool:
        """True if image embedding was present (not missing)."""
        if self.image_missing_mask is None:
            return True
        return not bool(self.image_missing_mask[item_id_idx].item())

In [321]:
# Paths (using the aligned artifacts)
image_embeddings_aligned_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")

# load id_maps for debugging (not required for runtime)
id_maps = {}
for name, path in [
    ("user_id_2idx", user_id_2idx_path),
    ("item_id_2idx", item_id_2idx_path),
    ("scene_2idx", scene_2idx_path),
]:
    with open(path, "r") as f:
        id_maps[name] = json.load(f)

feature_store = FeatureStore(
    text_embeddings_path=text_embeddings_path,
    image_embeddings_path=image_embeddings_aligned_path,
    time_features_path=time_features_path,
    df_id=df_id,
    image_missing_mask_path=image_missing_mask_path,
    id_maps=id_maps
)

print("FeatureStore created successfully.")
print("num_rows:", feature_store.num_rows)
print("text_dim:", feature_store.text_dim)
print("image_dim:", feature_store.image_dim)
print("time_dim:", feature_store.time_dim)

# --- Quick retrieval smoke test ---
test_row = 0
u_idx, i_idx, s_idx = feature_store.get_id_triplet(test_row)
text_vec = feature_store.get_text(test_row)
time_vec = feature_store.get_time(test_row)
img_vec = feature_store.get_image(i_idx)

print("\n--- Smoke Test ---")
print("row_idx:", test_row)
print("user_idx, item_idx, scene_idx:", (u_idx, i_idx, s_idx))
print("text_vec shape:", text_vec.shape)
print("time_vec shape:", tuple(time_vec.shape))
print("img_vec shape:", tuple(img_vec.shape))
print("has_image(item_idx):", feature_store.has_image(i_idx))

FeatureStore created successfully.
num_rows: 3730329
text_dim: 1536
image_dim: 768
time_dim: 5

--- Smoke Test ---
row_idx: 0
user_idx, item_idx, scene_idx: (41688, 13590, 4)
text_vec shape: (1536,)
time_vec shape: (5,)
img_vec shape: (768,)
has_image(item_idx): True


The FeatureStore class and the smoke test confirms that all modalities text embeddings, image embeddings, and time features are correctly indexed and retrievable using unified row- and item-based access.

This validates that the feature pipeline is fully aligned and ready for downstream fusion and model training without indexing or dimensional inconsistencies.

### ModalityAttentionFusion Implementation

Each modality (ID, text, image, time) is first projected into a shared representation space of dimension d_model. A learnable attention mechanism then computes per-sample weights over modalities, enabling the model to dynamically prioritize the most informative signals (e.g., text/image in cold-start settings and ID signals in warm-start settings).

The output is a single fused embedding of shape (B, d_model), along with stored attention weights for interpretability and debugging.


In [322]:
class ModalityAttentionFusion(nn.Module):
    """
    Projects {id, text, image, time} into a shared d_model space and applies
    per-sample attention over modalities.

    Forward inputs:
      id_vec:    (B, id_dim)
      text_vec:  (B, text_dim=1536)
      image_vec: (B, image_dim=768)
      time_vec:  (B, time_dim=5 or Time2Vec dim later)

    Output:
      fused: (B, d_model)

    Diagnostics:
      get_last_attention() returns a dict of attention weights.
    """
    def __init__(
        self,
        id_dim: int,
        text_dim: int,
        image_dim: int,
        time_dim: int,
        d_model: int = 256,
        attn_hidden: int = 128,
        dropout: float = 0.1
    ):
        super().__init__()

        # Projections into shared d_model
        self.proj_id = nn.Sequential(
            nn.Linear(id_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.proj_text = nn.Sequential(
            nn.Linear(text_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.proj_image = nn.Sequential(
            nn.Linear(image_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.proj_time = nn.Sequential(
            nn.Linear(time_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Attention MLP produces 1 scalar score per modality per sample
        self.attn_mlp = nn.Sequential(
            nn.Linear(d_model, attn_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(attn_hidden, 1)
        )

        self._last_attention: Optional[Dict[str, torch.Tensor]] = None

    def forward(
        self,
        id_vec: torch.Tensor,
        text_vec: torch.Tensor,
        image_vec: torch.Tensor,
        time_vec: torch.Tensor
    ) -> torch.Tensor:

        # Project each modality to (B, d_model)
        z_id = self.proj_id(id_vec)
        z_text = self.proj_text(text_vec)
        z_image = self.proj_image(image_vec)
        z_time = self.proj_time(time_vec)

        # Stack: (B, M=4, d_model)
        Z = torch.stack([z_id, z_text, z_image, z_time], dim=1)

        # Compute attention logits per modality: (B, 4, 1) -> (B, 4)
        logits = self.attn_mlp(Z).squeeze(-1)

        # Softmax over modalities -> (B, 4)
        attn = F.softmax(logits, dim=1)

        # Weighted sum: (B, 4, 1) * (B, 4, d_model) -> (B, d_model)
        fused = torch.sum(attn.unsqueeze(-1) * Z, dim=1)

        # Save for diagnostics (detach for safe logging)
        self._last_attention = {
            "id": attn[:, 0].detach(),
            "text": attn[:, 1].detach(),
            "image": attn[:, 2].detach(),
            "time": attn[:, 3].detach(),
        }

        return fused

    def get_last_attention(self) -> Optional[Dict[str, torch.Tensor]]:
        return self._last_attention

In [323]:
# ------- Minimal smoke test -------
# Choose dimensions based on the pipeline
text_dim = feature_store.text_dim     # 1536
image_dim = feature_store.image_dim   # 768
time_dim = feature_store.time_dim     # 5

# For now, choose an ID embedding dim (this will be finalize in Notebook 07)
id_dim = 64
d_model = 256

fusion = ModalityAttentionFusion(
    id_dim=id_dim,
    text_dim=text_dim,
    image_dim=image_dim,
    time_dim=time_dim,
    d_model=d_model,
    attn_hidden=128,
    dropout=0.1
).to(device)

B = 16
id_vec = torch.randn(B, id_dim).to(device)
text_vec = torch.randn(B, text_dim).to(device)
image_vec = torch.randn(B, image_dim).to(device)
time_vec = torch.randn(B, time_dim).to(device)

fused = fusion(id_vec, text_vec, image_vec, time_vec)

attn = fusion.get_last_attention()
attn_sum = attn["id"] + attn["text"] + attn["image"] + attn["time"]

print("Fused shape:", fused.shape)  # (B, d_model)
print("Attention sum (first 5):", attn_sum[:5].cpu().numpy())
print("Attention weights (first row):",
      {k: float(v[0].cpu()) for k, v in attn.items()})

Fused shape: torch.Size([16, 256])
Attention sum (first 5): [1.         0.99999994 1.         1.         1.        ]
Attention weights (first row): {'id': 0.2722589671611786, 'text': 0.25282910466194153, 'image': 0.24018172919750214, 'time': 0.23473016917705536}


In [324]:
# ----- “Real” smoke test using FeatureStore (small batch) -----
# small batch indices
batch_rows = np.random.randint(0, feature_store.num_rows, size=32)

# Creating modality tensors
text_batch = torch.tensor(np.stack([feature_store.get_text(i) for i in batch_rows]),
                          dtype=torch.float32).to(device)

time_batch = torch.stack([feature_store.get_time(i) for i in batch_rows]).to(device)

# Retrieve item_idx per row and fetch image embeddings
item_indices = [feature_store.get_id_triplet(i)[1] for i in batch_rows]
image_batch = torch.stack([feature_store.get_image(item_idx) for item_idx in item_indices]).to(device)

# Placeholder ID embedding vector (until IDEmbeddingLayer is added)
id_batch = torch.randn(len(batch_rows), id_dim).to(device)

fused_real = fusion(id_batch, text_batch, image_batch, time_batch)

attn_real = fusion.get_last_attention()
attn_sum_real = attn_real["id"] + attn_real["text"] + attn_real["image"] + attn_real["time"]

print("Real fused shape:", fused_real.shape)
print("Real attention sum mean:", float(attn_sum_real.mean().cpu()))
print("Real attention sum min/max:", float(attn_sum_real.min().cpu()), float(attn_sum_real.max().cpu()))

Real fused shape: torch.Size([32, 256])
Real attention sum mean: 1.0
Real attention sum min/max: 0.9999998807907104 1.0000001192092896


The fusion module successfully projects all modalities into a shared representation space and produces a fused embedding of the expected dimension. The attention weights across ID, text, image, and time modalities sum to one for each sample, confirming correct normalization and stable attention behaviour. The balanced distribution of weights indicates that the fusion mechanism is functioning as intended and is ready for scenario-aware routing.

### ScenarioAwareMMoE Implementation

This step implements the scenario-aware Mixture-of-Experts (MMoE) layer for ColdFusionX. Given the fused multimodal representation, the module routes each sample through multiple expert networks and uses a scenario-conditioned gating network (driven by scene_idx) to compute expert weights.

Key behavior:
*   Experts learn shared and specialized transformations of the fused embedding.
*   Gating depends on scene_idx, enabling scenario-specific routing and multi-scenario adaptation.
*   The model returns a combined expert output and exposes the latest gate weights for debugging and explainability.

In [325]:
class ScenarioAwareMMoE(nn.Module):
    """
    Scenario-aware Mixture-of-Experts (MMoE).

    Inputs:
      x:        (B, d_model)   fused representation
      scene_idx:(B,)           scenario indices (0..num_scenes-1)

    Outputs:
      y:        (B, hidden_dim) combined expert output

    Diagnostics:
      get_last_gates(): returns gate weights (B, num_experts)
    """
    def __init__(
        self,
        d_model: int,
        hidden_dim: int,
        num_experts: int,
        num_scenes: int,
        gate_embed_dim: int = 16,
        expert_dropout: float = 0.1
    ):
        super().__init__()

        self.d_model = d_model
        self.hidden_dim = hidden_dim
        self.num_experts = num_experts
        self.num_scenes = num_scenes

        # Experts: each expert maps (B, d_model) -> (B, hidden_dim)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, hidden_dim),
                nn.ReLU(),
                nn.Dropout(expert_dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(expert_dropout),
            )
            for _ in range(num_experts)
        ])

        # Scene embedding used for gating
        self.scene_embed = nn.Embedding(num_scenes, gate_embed_dim)

        # Gate network: conditioned on scene embedding (optionally we can include x too later)
        self.gate = nn.Sequential(
            nn.Linear(gate_embed_dim, gate_embed_dim),
            nn.ReLU(),
            nn.Linear(gate_embed_dim, num_experts)
        )

        self._last_gates: Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor, scene_idx: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, d_model)
            scene_idx: (B,)
        Returns:
            y: (B, hidden_dim)
        """
        B = x.size(0)

        # Expert outputs: list of (B, hidden_dim) -> stack -> (B, E, hidden_dim)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1)

        # Compute gate weights conditioned on scene_idx
        s_emb = self.scene_embed(scene_idx)           # (B, gate_embed_dim)
        gate_logits = self.gate(s_emb)                # (B, num_experts)
        gates = F.softmax(gate_logits, dim=-1)        # (B, num_experts)

        # Combine experts: (B, E, 1) * (B, E, hidden_dim) -> (B, hidden_dim)
        y = torch.sum(gates.unsqueeze(-1) * expert_outputs, dim=1)

        # Store for diagnostics
        self._last_gates = gates.detach()

        return y

    def get_last_gates(self) -> Optional[torch.Tensor]:
        return self._last_gates

In [326]:
# ------- Minimal smoke test -------
# Configuration
d_model = 256
hidden_dim = 256
num_experts = 4
num_scenes = scene_vocab_size  # 5 in AntM2C dataset

mmoe = ScenarioAwareMMoE(
    d_model=d_model,
    hidden_dim=hidden_dim,
    num_experts=num_experts,
    num_scenes=num_scenes,
    gate_embed_dim=16,
    expert_dropout=0.1
).to(device)

B = 16
x = torch.randn(B, d_model).to(device)
scene_idx = torch.randint(0, num_scenes, (B,)).to(device)

y = mmoe(x, scene_idx)
gates = mmoe.get_last_gates()

print("Input shape:", x.shape)
print("Output shape:", y.shape)              # (B, hidden_dim)
print("Gates shape:", gates.shape)           # (B, num_experts)
print("Gate sums (first 5):", gates.sum(dim=1)[:5].cpu().numpy())
print("First sample gate weights:", gates[0].cpu().numpy())

Input shape: torch.Size([16, 256])
Output shape: torch.Size([16, 256])
Gates shape: torch.Size([16, 4])
Gate sums (first 5): [0.9999999 0.9999999 1.0000001 1.        1.0000001]
First sample gate weights: [0.26030397 0.19586748 0.20643947 0.337389  ]


In [327]:
# “Real” smoke test using FeatureStore + Fusion output
# Sample real rows
batch_rows = np.random.randint(0, feature_store.num_rows, size=32)

# Retrieve scene_idx + item_idx
id_triplets = [feature_store.get_id_triplet(i) for i in batch_rows]
scene_batch = torch.tensor([t[2] for t in id_triplets], dtype=torch.long).to(device)
item_indices = [t[1] for t in id_triplets]

# Build modality tensors
text_batch = torch.tensor(np.stack([feature_store.get_text(i) for i in batch_rows]),
                          dtype=torch.float32).to(device)
time_batch = torch.stack([feature_store.get_time(i) for i in batch_rows]).to(device)
image_batch = torch.stack([feature_store.get_image(item_idx) for item_idx in item_indices]).to(device)

# Placeholder ID vectors until IDEmbeddingLayer is implemented
id_dim = 64
id_batch = torch.randn(len(batch_rows), id_dim).to(device)

# Use existing fusion module
fused_batch = fusion(id_batch, text_batch, image_batch, time_batch)  # (B, d_model)

# Pass through ScenarioAwareMMoE
mmoe_out = mmoe(fused_batch, scene_batch)  # (B, hidden_dim)
gates = mmoe.get_last_gates()

print("Fused batch shape:", fused_batch.shape)
print("MMoE output shape:", mmoe_out.shape)
print("Gates shape:", gates.shape)
print("Gate sum mean:", float(gates.sum(dim=1).mean().cpu()))
print("Gate sum min/max:", float(gates.sum(dim=1).min().cpu()), float(gates.sum(dim=1).max().cpu()))

Fused batch shape: torch.Size([32, 256])
MMoE output shape: torch.Size([32, 256])
Gates shape: torch.Size([32, 4])
Gate sum mean: 1.0
Gate sum min/max: 1.0 1.0000001192092896


The scenario-aware Mixture-of-Experts module correctly routes fused representations through multiple expert networks using scene-conditioned gating. The gate weights are well-formed, sum to one for each sample, and vary across experts, confirming effective scenario-specific routing. The output maintains the expected dimensionality, indicating that the module is stable and ready to be integrated into the full CTR prediction pipeline.

### Sanity Forward Pass (Fusion + Scenario Routing)

This section performs an end-to-end forward pass on a small batch to validate:
1.   Modality retrieval is aligned
2.   Attention fusion produces stable outputs with attention weights summing to 1
3.   Scenario-aware MMoE gating produces valid probability weights summing to 1
4.   The final outputs contain no NaNs/Infs and match expected tensor shapes

In [339]:
# Set sanity-check hyperparameters
SANITY_BATCH_SIZE = 64

ID_DIM = 256        # must match output of SimpleIDEmbedding
D_MODEL = 256       # fusion output dim
HIDDEN_DIM = 256    # MMoE output dim

NUM_EXPERTS = 4
NUM_SCENES = scene_vocab_size

In [340]:
class SimpleIDEmbedding(nn.Module):
    def __init__(self, num_users, num_items, num_scenes, user_dim, item_dim, scene_dim, out_dim):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, user_dim)
        self.item_emb = nn.Embedding(num_items, item_dim)
        self.scene_emb = nn.Embedding(num_scenes, scene_dim)
        self.proj = nn.Linear(user_dim + item_dim + scene_dim, out_dim)

    def forward(self, user_idx, item_idx, scene_idx):
        u = self.user_emb(user_idx)
        i = self.item_emb(item_idx)
        s = self.scene_emb(scene_idx)
        x = torch.cat([u, i, s], dim=1)
        return self.proj(x)

In [330]:
id_encoder_sanity = SimpleIDEmbedding(
    num_users=user_vocab_size,
    num_items=item_vocab_size,
    num_scenes=scene_vocab_size,
    user_dim=64,
    item_dim=64,
    scene_dim=32,
    out_dim=ID_DIM
).to(device).eval()

In [342]:
# Create new ModalityAttentionFusion + ScenarioAwareMMoE objects
fusion_sanity = ModalityAttentionFusion(
    id_dim=ID_DIM,                 # this must match id_vec dim
    text_dim=feature_store.text_dim,  # 1536
    image_dim=feature_store.image_dim, # 768
    time_dim=feature_store.time_dim,   # 5
    d_model=D_MODEL,
    attn_hidden=128,
    dropout=0.1
).to(device).eval()

mmoe_sanity = ScenarioAwareMMoE(
    d_model=D_MODEL,
    hidden_dim=HIDDEN_DIM,
    num_experts=NUM_EXPERTS,
    num_scenes=NUM_SCENES,
    gate_embed_dim=16,
    expert_dropout=0.1
).to(device).eval()

Forward pass + ALL sanity checks

In [343]:
# --- Sample batch from df_id (preserve original row indices for alignment) ---
batch_df = df_id.sample(n=SANITY_BATCH_SIZE, random_state=42)
row_indices = batch_df.index.to_numpy()  # IMPORTANT: these align with text/time tensors

In [344]:
scene_col = "scene_idx"

user_idx_batch = torch.tensor(batch_df["user_id_idx"].to_numpy(), dtype=torch.long)
item_idx_batch = torch.tensor(batch_df["item_id_idx"].to_numpy(), dtype=torch.long)
scene_idx_batch = torch.tensor(batch_df[scene_col].to_numpy(), dtype=torch.long)

# --- Build modality tensors using FeatureStore ---
text_batch = torch.tensor(
    np.stack([feature_store.get_text(int(r)) for r in row_indices]),
    dtype=torch.float32
)
time_batch = torch.stack([feature_store.get_time(int(r)) for r in row_indices]).float()
image_batch = torch.stack([feature_store.get_image(int(i)) for i in item_idx_batch.tolist()]).float()

# Move to device
text_batch = text_batch.to(device)
time_batch = time_batch.to(device)
image_batch = image_batch.to(device)
user_idx_batch = user_idx_batch.to(device)
item_idx_batch = item_idx_batch.to(device)
scene_idx_batch = scene_idx_batch.to(device)

# --- Forward: ID -> Fusion -> MMoE ---
with torch.no_grad():
    id_vec = id_encoder_sanity(user_idx_batch, item_idx_batch, scene_idx_batch)  # (B, ID_DIM)
    fused = fusion_sanity(id_vec, text_batch, image_batch, time_batch)           # (B, D_MODEL)
    attn = fusion_sanity.get_last_attention()                                    # dict: (B,)
    routed = mmoe_sanity(fused, scene_idx_batch)                                 # (B, HIDDEN_DIM)
    gates = mmoe_sanity.get_last_gates()                                         # (B, NUM_EXPERTS)

In [334]:
# --- Sanity checks ---
def assert_no_nan_inf(x, name):
    assert torch.isfinite(x).all(), f"{name} contains NaN/Inf"

assert_no_nan_inf(id_vec, "id_vec")
assert_no_nan_inf(fused, "fused")
assert_no_nan_inf(routed, "routed")
assert_no_nan_inf(gates, "gates")

# Attention sums
attn_sum = attn["id"] + attn["text"] + attn["image"] + attn["time"]
assert torch.allclose(attn_sum, torch.ones_like(attn_sum), atol=1e-5), "Attention weights do not sum to 1"

# Gate sums
gate_sum = gates.sum(dim=1)
assert torch.allclose(gate_sum, torch.ones_like(gate_sum), atol=1e-6), "Gate weights do not sum to 1"

print("Sanity forward pass successful.")
print("id_vec:", id_vec.shape)
print("fused:", fused.shape)
print("routed:", routed.shape)
print("gates:", gates.shape)
print("Attention sum min/max:", float(attn_sum.min().cpu()), float(attn_sum.max().cpu()))
print("Gate sum min/max:", float(gate_sum.min().cpu()), float(gate_sum.max().cpu()))

Sanity forward pass successful.
id_vec: torch.Size([64, 256])
fused: torch.Size([64, 256])
routed: torch.Size([64, 256])
gates: torch.Size([64, 4])
Attention sum min/max: 0.9999998807907104 1.0000001192092896
Gate sum min/max: 1.0 1.0000001192092896


The end-to-end sanity forward pass confirms that interaction-level and item-level features are correctly aligned and retrievable, and that the fusion and scenario-aware routing components produce stable outputs with properly normalized attention and gating weights. This validates the core feature-to-representation pipeline and indicates readiness for downstream training and evaluation.

### Saving Fusion Artifacts

This step stores lightweight artifacts required for reproducibility and downstream notebooks:


*   Fusion_config.json captures key model dimensions and architectural hyperparameters used in the fusion and scenario-routing blocks.
*   sample_batch_debug.pt optionally stores a small batch of tensors (IDs + modalities) used in the sanity forward pass, allowing later notebooks to reproduce behavior without re-sampling.

These outputs support consistent experimentation, debugging, and report traceability across the ColdFusionX pipeline.

In [349]:
fusion_config = {
    # Dataset-level metadata
    "num_users": int(user_vocab_size),
    "num_items": int(item_vocab_size),
    "num_scenes": int(scene_vocab_size),

    # Modality dimensions (from FeatureStore)
    "text_dim": int(feature_store.text_dim),      # 1536
    "image_dim": int(feature_store.image_dim),    # 768
    "time_dim": int(feature_store.time_dim),      # 5 (raw) - may change when Time2Vec is used

    # Fusion block parameters (sanity version or chosen defaults)
    "id_dim": int(ID_DIM),
    "d_model": int(D_MODEL),
    "attn_hidden": 128,
    "fusion_dropout": 0.1,

    # MMoE parameters
    "hidden_dim": int(HIDDEN_DIM),
    "num_experts": int(NUM_EXPERTS),
    "gate_embed_dim": 16,
    "expert_dropout": 0.1,

    # Notes / provenance
    "image_embeddings_aligned": True,
    "image_embeddings_file": "image_embeddings_vit_base_aligned.pt",
    "missing_image_mask_file": "image_missing_mask.pt",
    "text_embeddings_file": "text_embeddings.npy",
    "time_features_file": "time_features_final.pt"
}

In [350]:
fusion_config_path = os.path.join(fusion_artifacts_dir, "fusion_config.json")
with open(fusion_config_path, "w") as f:
    json.dump(fusion_config, f, indent=2)

print("Saved fusion_config.json to:", fusion_config_path)

Saved fusion_config.json to: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts/fusion_config.json


#### Save sample_batch_debug.pt
This stores the exact tensors used. Keeping it small (64 rows only) to avoid storage bloat.

In [351]:
sample_debug = {
    "row_indices": row_indices,  # numpy array of interaction indices (alignment reference)

    # ID indices
    "user_id_idx": user_idx_batch.detach().cpu(),
    "item_id_idx": item_idx_batch.detach().cpu(),
    "scene_idx": scene_idx_batch.detach().cpu(),

    # Modalities (CPU)
    "text_batch": text_batch.detach().cpu(),
    "image_batch": image_batch.detach().cpu(),
    "time_batch": time_batch.detach().cpu(),

    # Model outputs (CPU)
    "id_vec": id_vec.detach().cpu(),
    "fused": fused.detach().cpu(),
    "routed": routed.detach().cpu(),
    "gates": gates.detach().cpu(),
    "attention": {
        "id": fusion_sanity.get_last_attention()["id"].detach().cpu(),
        "text": fusion_sanity.get_last_attention()["text"].detach().cpu(),
        "image": fusion_sanity.get_last_attention()["image"].detach().cpu(),
        "time": fusion_sanity.get_last_attention()["time"].detach().cpu(),
    }
}

The debug batch is especially useful when:
*   Gating/attention looks unstable during training
*   SHAP explanations need deterministic sample selection
*   Demonstrate reproducibility in the report

In [354]:
debug_batch_path = os.path.join(fusion_artifacts_dir, "sample_batch_debug.pt")
torch.save(sample_debug, debug_batch_path)

print("Saved sample_batch_debug.pt to:", debug_batch_path)
print("Debug batch keys:", list(sample_debug.keys()))

Saved sample_batch_debug.pt to: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts/sample_batch_debug.pt
Debug batch keys: ['row_indices', 'user_id_idx', 'item_id_idx', 'scene_idx', 'text_batch', 'image_batch', 'time_batch', 'id_vec', 'fused', 'routed', 'gates', 'attention']


In [356]:
loaded = torch.load(debug_batch_path, map_location="cpu", weights_only=False)

In [357]:
print("Loaded debug batch successfully.")
print("Keys:", list(loaded.keys()))
print("text_batch:", loaded["text_batch"].shape)
print("image_batch:", loaded["image_batch"].shape)
print("time_batch:", loaded["time_batch"].shape)
print("gates:", loaded["gates"].shape)

Loaded debug batch successfully.
Keys: ['row_indices', 'user_id_idx', 'item_id_idx', 'scene_idx', 'text_batch', 'image_batch', 'time_batch', 'id_vec', 'fused', 'routed', 'gates', 'attention']
text_batch: torch.Size([64, 1536])
image_batch: torch.Size([64, 768])
time_batch: torch.Size([64, 5])
gates: torch.Size([64, 4])
